# Create FLUT

The test requires a non-reactive FLUT with variable enthalpy/temperature and constant mixture.

In [ ]:
import pyFLUT
import cantera as ct
import numpy as np

Load mechanism (included in cantera)

In [ ]:
gas = ct.Solution('ch4_smooke_EKT.yaml')

### Create manifold

In [ ]:
# Temperatures
Ts = np.linspace(300,800,21)

# Nitrogen mass fractions
Y_N2 = 1

Create solution array

In [ ]:
sol_arr = ct.SolutionArray(gas,len(Ts))

In [ ]:
species_array = np.zeros((len(Ts),gas.n_species))
species_array[:,gas.species_index('N2')] = Y_N2

In [ ]:
sol_arr.TPY = Ts, ct.one_atm, species_array

### Build flut from the data

In [ ]:
data = np.concatenate((sol_arr.T[:,np.newaxis],sol_arr.P[:,np.newaxis],sol_arr.density[:,np.newaxis],sol_arr.enthalpy_mass[:,np.newaxis],species_array),axis=1)

In [ ]:
import collections

input_dict = collections.OrderedDict({'T':sol_arr.T})
output_list = [(name,i) for i, name in enumerate(['T','p','rho','ha'])]
output_list = output_list + [(name, i + len(output_list)) for i, name in enumerate(gas.species_names)]
output_dict = collections.OrderedDict(output_list)

flut = pyFLUT.Flut(data,input_dict=input_dict,output_dict=output_dict)
flut

In [ ]:
flut.calc_non_dimensional_quantity(along_variable='T',quantity='ha')
flut = flut.map_variables(from_inp='T',to_inp='hanorm')
flut

Calc additionally needed variables and rename (maybe this can be adapted in the future)

In [ ]:
flut.calc_with_cantera(['rho', 'cp', 'lambda', 'visc','alpha'],gas=gas)
flut = flut.rename_variables(rename_dict={'visc':'mu'})

In [ ]:
flut.output_dict

In [ ]:
output_variables = ['T','p','rho','ha','O2','N2','hanorm','ha_max','ha_min','cp','lambda','mu','alpha']
flut.export_to_openfoam('single_species_Tnorm_flut.h5',cantera_file='ch4_smooke_EKT.yaml',output_variables=output_variables)

### Check 

In [ ]:
read_flut = pyFLUT.Flut.read_h5('single_species_Tnorm_flut.h5')

In [ ]:
read_flut.output_dict

### Generate profile

Heating profile. 300 + 125 + 225.

In [ ]:
indices = [0,5,15]
sol_arr.T[indices]

In [ ]:
enthalpies = sol_arr.enthalpy_mass[indices]
enthalpies